# Baseline results


In [10]:
import os
import random
import warnings
from dotenv import load_dotenv
from rich import pretty, print, traceback
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
traceback.install(show_locals=True)
pretty.install()
load_dotenv();

In [11]:
positive_class = "Pneumonia"

## Load model


In [12]:
from model_chexagent.chexagent import CheXagent
chexagent = CheXagent()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.00s/it]


## Load dataset


In [13]:
from datasets import load_dataset, Dataset
ds: Dataset = load_dataset("vllm-pneumonia-detection/mimic-500-gt", split="test")
image_root_path = "/home/nicoleg/physionet.org/files/mimic-cxr-jpg/2.0.0/"

def in_domain(sample):
    label = sample[positive_class]
    return label == 0 or label == 1

def set_image_path(sample):
    sample["image_files"] = image_root_path + sample["image_files"]
    return sample

ds = ds.filter(in_domain)
ds = ds.map(set_image_path)

In [14]:
sample = random.choice(ds)
print(sample)

{
    'dicom_id': '99a719f1-338c19ff-4c6100c3-a98e761a-254572ee',
    'split': 'test',
    'study_id': '58245185',
    'subject_id': '17720924',
    'image_files': 
'/home/nicoleg/physionet.org/files/mimic-cxr-jpg/2.0.0/files/p17/p17720924/s58245185/99a719f1-338c19ff-4c6100c3-a98
e761a-254572ee.jpg',
    'reports': '                                 FINAL REPORT\n CHEST RADIOGRAPH\n \n TECHNIQUE:  PA and lateral 
chest views were reviewed in comparison with prior\n chest radiographs with the most recent from ___.\n \n 
FINDINGS:  Lungs are clear. No evidence of pulmonary edema or pneumonia. Focal\n opacity over anatomical region of 
lingula which is perceived only on frontal\n view represents a pericardial fat.  Heart size, mediastinal and hilar 
contours\n are normal.  There is no pleural effusion or pneumothorax.  \n \n IMPRESSION; No pulmonary edema\n',
    'Atelectasis': -2.0,
    'Cardiomegaly': 0.0,
    'Consolidation': -2.0,
    'Edema': 0.0,
    'Enlarged Cardiomediastinum': 0.0,
    'Fracture': -2.0,
    'Lung Lesion': -2.0,
    'Lung Opacity': -2.0,
    'No Finding': -2.0,
    'Effusion': 0.0,
    'Pleural Other': -2.0,
    'Pneumonia': 0.0,
    'Pneumothorax': 0.0,
    'Support Devices': -2.0
}

# Get results


In [15]:
def get_response_with_report(paths: list, report: str) -> str:
    prompt = f'{report.strip()}\nDoes this chest X-ray contain a {positive_class}?'
    response = chexagent.generate(paths, prompt)
    return response

results = {}
yes_no = {"yes": 1, "no": 0}
for sample in tqdm(ds.shuffle(seed=42)):
    dicom_id = sample["dicom_id"]
    path = sample["image_files"]
    report = sample["reports"]
    response = get_response_with_report([path], report).lower()
    ypred = yes_no[response]
    results[dicom_id] = {
        "y_true": sample[positive_class],
        "y_pred": ypred,
        "image_files": path,
    }

  0%|          | 0/89 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  1%|          | 1/89 [00:01<02:40,  1.83s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  2%|▏         | 2/89 [00:03<02:14,  1.55s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  3%|▎         | 3/89 [00:04<02:09,  1.50s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  4%|▍         | 4/89 [00:05<02:00,  1.42s/it]The attention mask and the pad token id were not set. As a cons

In [16]:
import pandas as pd

df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "dicom_id"
df.to_csv("results_with_report.csv")

df

,y_true,y_pred,image_files
dicom_id,,,
4a5bbca6-64ed6abf-84645068-6a7688bd-9a9910d4,1.0,1,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
7b43b8ff-190d3ca9-03cfbbd3-45ad3d0d-72d06c1c,1.0,1,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
463d2a28-b411bb98-f7bda38e-7030ebb9-74a8a1e0,0.0,0,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
a6a6000b-26fecffe-b90ded19-73b8a48f-6e3f7557,0.0,0,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
0cdfb937-27e0834d-4d8d1c40-cee9e187-98390c95,0.0,0,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
...,...,...,...
7f83f5d5-3afe2911-3b666b80-5dbde6e1-f2a9d980,0.0,0,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
6a7b83c9-7b7c6ba9-09d85de8-a76f1aa7-4fd0e047,1.0,1,/home/nicoleg/physionet.org/files/mimic-cxr-jp...
54a9e5bc-2d3b9e9a-43c44b54-7c16e7b1-f923f86c,1.0,1,/home/nicoleg/physionet.org/files/mimic-cxr-jp...


In [17]:
y_true = df["y_true"].values
y_pred = df["y_pred"].values
target_names = [f"No {positive_class}", positive_class]


def generate_classification_report(
    y_true, y_pred, target_names=[f"No Pneumonia", "Pneumonia"]
):
    """
    Generate a classification report and confusion matrix.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    target_names : list
        List of class names

    Returns:
    --------
    cr : dict
        Classification report as a dictionary
    """
    from sklearn.metrics import classification_report, confusion_matrix
    from tabulate import tabulate

    # Classification report
    cr_string = classification_report(y_true, y_pred, target_names=target_names)
    cr = classification_report(
        y_true, y_pred, target_names=target_names, output_dict=True
    )
    print("Classification Report:")
    print(cr_string)

    # Confusion matrix
    print("Confusion Matrix:")
    cfm = confusion_matrix(y_true, y_pred)
    print(
        tabulate(
            cfm, headers=target_names, showindex=target_names, tablefmt="rounded_grid"
        )
    )
    return cr


cr = generate_classification_report(
    y_true,
    y_pred,
    target_names=target_names,
)

Classification Report:

precision    recall  f1-score   support

No Pneumonia       0.95      0.93      0.94        59
   Pneumonia       0.87      0.90      0.89        30

    accuracy                           0.92        89
   macro avg       0.91      0.92      0.91        89
weighted avg       0.92      0.92      0.92        89

Confusion Matrix:

╭──────────────┬────────────────┬─────────────╮
│              │   No Pneumonia │   Pneumonia │
├──────────────┼────────────────┼─────────────┤
│ No Pneumonia │             55 │           4 │
├──────────────┼────────────────┼─────────────┤
│ Pneumonia    │              3 │          27 │
╰──────────────┴────────────────┴─────────────╯

In [18]:
summary = pd.DataFrame(
    {
        "f1-score": round(cr["macro avg"]["f1-score"], 4),
        # "roc auc": round(roc_auc_score(y_true, y_pred), 4),
        "sensitivity": round(cr["Pneumonia"]["recall"], 4),
        "specificity": round(cr["No Pneumonia"]["recall"], 4),
        "recall": round(cr["macro avg"]["recall"], 4),
        "precision": round(cr["macro avg"]["precision"], 4),
        "accuracy": round(cr["accuracy"], 4),
    },
    index=["chexagent"],
)
summary

,f1-score,sensitivity,specificity,recall,precision,accuracy
chexagent,0.9127,0.9,0.9322,0.9161,0.9096,0.9213
